# 20261106 · LangGraph 进阶（工具节点 / 记忆 / 运行时 / 人机协作 / 时间旅行）

> 对应课堂脚本：`week24/20261106/test1.py` ～ `test9.py`
>
> 上一讲（`20261104_LangGraph.ipynb`）讲的是**图长什么样**：State、边、reducer、子图、Send、基础检查点。
> 这一讲讲的是**图跑起来之后靠什么活着**：工具到底谁执行、记忆存哪、上下文怎么注入、结果怎么吐出去、人什么时候接管、能不能回到过去重来。

## 主线

| 阶段 | 小节 | 对应脚本 | 解决什么 |
|---|---|---|---|
| 执行 | 1. ToolNode | test1 | 模型「申请」调用工具，节点「执行」工具——两件事分开 |
| 记忆 | 2～3. Store | test2 / test3 | 跨会话的长期事实放这里，和检查点是两套存储 |
| 上下文 | 4. Runtime / Context | test4 | 用户身份、配置、store 怎么传进节点 |
| 上下文 | 5. ToolRuntime | test5 | 工具内部也能拿到运行时（做权限校验） |
| 输出 | 6. stream_mode | test6 | 结果怎么一点点吐给前端 |
| 接管 | 7. interrupt_before | test7 | 节点执行**前**停下来等人 |
| 接管 | 8. interrupt() | test8 | 节点执行**到某一行**停下来等人 |
| 反悔 | 9. 时间旅行 | test9 | 回到任意历史点重放，或改一笔走新分支 |

## 证据边界

- 所有代码格都在本机 `langgraph 1.2.18` + `langchain 1.4.7` 上**实机执行过**，输出就是真实结果。
- 涉及模型的格子走 `week23/.env` 里的千问配置；课堂脚本中硬编码的 API Key 不在本笔记复现。
- 需要**确定性**的章节（工具循环结构、断点、时间旅行）用「剧本模型」代替真模型，保证每次输出一致。


## 0. 环境与统一模型配置

课堂 9 个脚本里，`api_key` 和 `base_url` 是**逐文件硬编码**的（同一把 `sk-ws-...` Key 复制了 9 遍）。本笔记统一改走 week23 的配置：

| 项 | 值 |
|---|---|
| 配置文件 | `week23/.env`（`LLM_PROVIDER=qwen`） |
| 读取入口 | `week23/20261027/lab_config.py` 的 `get_model()` |
| 模型 | `qwen3.8-flash`（OpenAI 兼容模式，关闭 thinking） |
| 内核 | `langchain-learning (3.11.9)`，跑在 `week23/.venv` |

> 工程习惯：**Key 只该出现在一处**。课堂代码把它们复制了 9 份，等于把泄漏面乘了 9；真上线时这属于必须修的安全债。


In [1]:
import os, sys, json, uuid, pathlib
from dataclasses import dataclass
from typing import Annotated, TypedDict, Any
from importlib.metadata import version


def find_lab_dir(start: str | None = None) -> pathlib.Path:
    """从当前目录逐级往上找 week23/20261027/lab_config.py。"""
    root = pathlib.Path(start or os.getcwd()).resolve()
    for cand in [root, *root.parents]:
        target = cand / "week23" / "20261027" / "lab_config.py"
        if target.exists():
            return target.parent
    raise FileNotFoundError("没找到 week23/20261027/lab_config.py，请从 week24 目录启动内核")


LAB_DIR = find_lab_dir()
sys.path.insert(0, str(LAB_DIR))
from lab_config import get_model, show_config

print("lab_config    :", LAB_DIR)
print("langgraph     :", version("langgraph"))
print("langchain     :", version("langchain"))
print("langchain_core:", version("langchain_core"))
print("-" * 62)
show_config()


lab_config    : D:\project\step1\week23\20261027
langgraph     : 1.2.18
langchain     : 1.4.7
langchain_core: 1.7.0
--------------------------------------------------------------
provider = qwen
model    = qwen3.8-flash
base_url = https://cn-beijing.maas.qianwenaiapi.com/compatible-mode/v1
api_key  = sk-ws-***ef-t
max_tokens = 1024
thinking   = False（true 会白烧 reasoning token）


### 0.1 本笔记里的两个「模型」

| 名字 | 是什么 | 用在哪几节 |
|---|---|---|
| `model` | 真模型（千问 `qwen3.8-flash`） | 自检、1、5、6：需要真实的工具选择与 token 流 |
| `ScriptedModel` | 剧本模型：第一轮固定申请调工具，看到 `ToolMessage` 就给结论 | 1、7、9：需要**确定性**，否则断点与时间旅行的断言无法复现 |

`langchain_core` 自带的 `GenericFakeChatModel` **不支持 `bind_tools`**（实测抛 `NotImplementedError`），所以这里手写一个最小替身——它只需要实现 `bind_tools` 和 `invoke` 两个方法。


In [2]:
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage

model = get_model()          # 真模型


class ScriptedModel:
    """剧本模型：只实现节点真正会用到的那两个方法。"""

    def __init__(self, tool_name: str = "get_weather", args: dict | None = None,
                 final: str = "（剧本模型）已按工具结果作答。"):
        self.tool_name = tool_name
        self.args = args or {"city": "Tokyo"}
        self.final = final
        self.turns = 0

    def bind_tools(self, tools, **kwargs):        # 真模型才需要绑定，这里原样返回
        return self

    def invoke(self, messages, **kwargs):
        self.turns += 1
        tool_msgs = [m for m in messages if isinstance(m, ToolMessage)]
        if tool_msgs:                             # 已经拿到工具结果 → 收尾
            return AIMessage(content=f"{self.final}工具返回：{tool_msgs[-1].content}")
        return AIMessage(                          # 第一轮 → 申请调用工具
            content="",
            tool_calls=[{"name": self.tool_name, "args": self.args, "id": f"call_{self.turns}"}],
        )


def make_scripted_chat(tool_name: str = "get_weather", args: dict | None = None):
    """把剧本模型包成节点函数，供后面几节构建确定性图。"""
    llm = ScriptedModel(tool_name, args)

    def chat_fn(state):
        return {"messages": [llm.invoke(state["messages"])]}

    return chat_fn


print("真模型 :", type(model).__name__)
print("剧本模型:", type(ScriptedModel()).__name__)


真模型 : ChatOpenAI
剧本模型: ScriptedModel


In [3]:
import time
from langchain.tools import tool


@tool
def get_weather(city: str) -> str:
    """查询指定城市的天气"""
    return f"{city} 今天晴，25℃"


t0 = time.perf_counter()
reply = model.invoke([{"role": "user", "content": "用一句话说明 LangGraph 是什么"}])
print(f"[纯文本  ] {time.perf_counter() - t0:.2f}s → {reply.content[:90]!r}")

t0 = time.perf_counter()
picked = model.bind_tools([get_weather]).invoke([{"role": "user", "content": "东京今天天气怎么样？"}])
print(f"[工具选择] {time.perf_counter() - t0:.2f}s → tool_calls = {picked.tool_calls}")


[纯文本  ] 1.14s → 'LangGraph 是一个用于构建有状态、多参与者 LLM 应用程序的框架，它通过图结构来编排和控制复杂的代理工作流。'
[工具选择] 0.62s → tool_calls = [{'name': 'get_weather', 'args': {'city': '东京'}, 'id': 'call_fb79657f2fa745c6b5205693', 'type': 'tool_call'}]


## 1. ToolNode：把「申请调用」和「真执行」拆成两个节点

1102 那节我们手写过 ReAct 循环：自己解析 `tool_calls`、自己调函数、自己拼 `ToolMessage` 塞回消息列表。LangGraph 把这套循环固化成两个原语：

```text
chat 节点       模型返回 AIMessage(tool_calls=[...])        ← 只是「申请书」
tool_node 节点   按 tool_calls 逐个执行，产出 ToolMessage    ← 真正「干活」
```

循环条件由**条件边**判断：最后一条消息有 `tool_calls` → 去 `tool_node`，没有 → `END`；`tool_node` 再静态连回 `chat`，ReAct 循环就闭合了。

```mermaid
flowchart LR
    S((START)) --> C[chat]
    C -->|有 tool_calls| T[tool_node]
    C -->|没有| E((END))
    T --> C
```

> 顺带一个环境提示：`get_graph().draw_ascii()` 需要额外的 `grandalf` 包（本机未装，实测报 `ImportError`）；`draw_mermaid_png()` 要联网。本笔记改用「节点 + 边」清单打印，零依赖。


In [4]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode


tools = [get_weather]
QUESTION = "请调用工具查一下东京今天的天气，再用一句话总结"
llm_with_tools = model.bind_tools(tools)      # 真模型：自己决定调不调工具
tool_node = ToolNode(tools)


def chat(state: MessagesState):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}


def route(state: MessagesState):
    return "tool_node" if state["messages"][-1].tool_calls else END


builder = StateGraph(MessagesState)
builder.add_node("chat", chat)
builder.add_node("tool_node", tool_node)
builder.add_edge(START, "chat")
builder.add_conditional_edges("chat", route, ["tool_node", END])
builder.add_edge("tool_node", "chat")
graph1 = builder.compile()

g = graph1.get_graph()
print("节点：", list(g.nodes))
for e in g.edges:
    kind = "条件边" if getattr(e, "conditional", False) else "普通边"
    print(f"边  ： {e.source:>10s} → {e.target:<10s} [{kind}]")


节点： ['__start__', 'chat', 'tool_node', '__end__']
边  ：  __start__ → chat       [普通边]
边  ：       chat → __end__    [条件边]
边  ：       chat → tool_node  [条件边]
边  ：  tool_node → chat       [普通边]


In [5]:
for chunk in graph1.stream({"messages": [("user", QUESTION)]}, stream_mode="updates"):
    for node, update in chunk.items():
        print(f"【{node}】")
        for m in update.get("messages", []):
            print(f"  {type(m).__name__:11s} content={m.content!r}")
            if getattr(m, "tool_calls", None):
                print(f"              tool_calls={m.tool_calls}")
            if getattr(m, "tool_call_id", None):
                print(f"              tool_call_id={m.tool_call_id}")


【chat】
  AIMessage   content=''
              tool_calls=[{'name': 'get_weather', 'args': {'city': '东京'}, 'id': 'call_43615ab07d124c7185a61ff7', 'type': 'tool_call'}]
【tool_node】
  ToolMessage content='东京 今天晴，25℃'
              tool_call_id=call_43615ab07d124c7185a61ff7
【chat】
  AIMessage   content='东京今天天气晴朗，气温为25℃。'


### 知识点解析 1：ToolNode 到底做了什么

| 关注点 | 说明 |
|---|---|
| 输入 | state 里的 `messages` 键（可用 `ToolNode(..., messages_key=...)` 改） |
| 取什么 | 最后一条 `AIMessage.tool_calls`；一次多个则**并行执行** |
| 输出 | `ToolMessage(content=..., tool_call_id=..., name=...)` 列表 |
| 模型看到什么 | 只有工具名 + 参数 schema；函数体、`runtime` 参数、返回类型注解都看不到 |
| 工具抛异常 | 默认转成一条 `ToolMessage` 把错误回给模型（`handle_tool_errors` 可配），不是直接崩图 |
| 为什么拆两个节点 | 执行前后都能插检查点、插人审、插日志；模型只管「想」，节点负责「做」 |

**课堂 `test1.py` 有一处冗余**：它在 `chat` 上同时挂了条件边和静态边：

```python
builder.add_conditional_edges("chat", condition_edge)
builder.add_edge("chat", END)      # ← 多余
```

下一格用同一张图跑两种写法，看看到底有没有区别。


In [6]:
def build_demo(extra_edge_to_end: bool):
    turns = {"n": 0}

    def chat_scripted(state):
        turns["n"] += 1
        if turns["n"] == 1:
            return {"messages": [AIMessage(content="", tool_calls=[
                {"name": "get_weather", "args": {"city": "Tokyo"}, "id": "call_1"}])]}
        return {"messages": [AIMessage(content="done")]}

    b = StateGraph(MessagesState)
    b.add_node("chat", chat_scripted)
    b.add_node("tool_node", ToolNode(tools))
    b.add_edge(START, "chat")
    b.add_conditional_edges(
        "chat",
        lambda s: "tool_node" if s["messages"][-1].tool_calls else END,
        ["tool_node", END],
    )
    b.add_edge("tool_node", "chat")
    if extra_edge_to_end:
        b.add_edge("chat", END)          # 课堂 test1 的写法
    return b.compile(), turns


for flag in (False, True):
    g, turns = build_demo(flag)
    out = g.invoke({"messages": [HumanMessage("hi")]})
    print(f"add_edge('chat', END) = {str(flag):5s} | chat 调用 {turns['n']} 次 | "
          f"消息 {len(out['messages'])} 条 | 末条 = {out['messages'][-1].content!r}")


add_edge('chat', END) = False | chat 调用 2 次 | 消息 4 条 | 末条 = 'done'
add_edge('chat', END) = True  | chat 调用 2 次 | 消息 4 条 | 末条 = 'done'


### 实测结论（对应上面输出）

| 写法 | chat 调用次数 | 结果 |
|---|---|---|
| 只有条件边 | 2 | 正常：chat → tool_node → chat → END |
| 条件边 + `add_edge("chat", END)` | 2 | **也能跑通，但语义含混** |

结论：在 `langgraph 1.2.18` 里，同一节点上「静态边 + 条件边」并存不会报错，条件边仍然生效。
但它是**隐性陷阱**——静态边意味着「chat 之后无条件也要去 END」，一旦你后面想把条件边的某个分支改成「跳过 END 继续跑」，这条静态边会让流程在同一个 superstep 里多激活一个终点，症状是「莫名其妙提前结束」。

> 规则：**一个节点的出边，静态与条件二选一**。要路由就只写条件边。


## 2. Store：跨会话的长期记忆

检查点（checkpointer）和 Store 是**两套存储**，混用会出问题：

| | checkpointer | Store |
|---|---|---|
| 存什么 | 整条执行轨迹：每个 superstep 的 state 快照 | 键值事实：用户画像、偏好、历史结论 |
| 隔离维度 | `thread_id`（一次会话） | `namespace`（如 `("memories", user_id)`） |
| 主要用途 | 多轮对话、断点恢复、时间旅行 | 换个会话还记得「这个人是谁」 |
| 检索方式 | 按 checkpoint 顺序取 | `get` 精确取 + `search` 相关度排序 |
| 典型实现 | `InMemorySaver` / `RedisSaver` | `InMemoryStore` / `PostgresStore` |

课堂 `test2.py` 的注释写着「Store 内部会对它做 embedding，然后按相似度排序返回」。**这句话默认不成立**，下面两格直接对比。


In [7]:
from langgraph.store.memory import InMemoryStore

ns = ("memories", "user_1")
plain_store = InMemoryStore()
print("index_config =", plain_store.index_config, "| embeddings =", plain_store.embeddings)

for text in ["今天成都天气晴", "User name is Bob", "User likes Python"]:
    plain_store.put(ns, str(uuid.uuid4()), {"value": text})

print("\n默认 store 上 query='Bob likes python' 的检索结果：")
for item in plain_store.search(ns, query="Bob likes python"):
    print(f"  score={item.score!s:>6s}  {item.value['value']}")


index_config = None | embeddings = None

默认 store 上 query='Bob likes python' 的检索结果：
  score=  None  今天成都天气晴
  score=  None  User name is Bob
  score=  None  User likes Python


In [8]:
def letter_embed(texts: list[str]) -> list[list[float]]:
    """把 a-z 的字母频次当向量（26 维）。中文没有字母 → 全 0，分数自然低。
    故意选这么土的 embedding，是为了让「语义检索」可被手算验证，而不是黑箱。"""
    out = []
    for t in texts:
        v = [0.0] * 26
        for ch in t.lower():
            if "a" <= ch <= "z":
                v[ord(ch) - 97] += 1.0
        out.append(v)
    return out


vector_store = InMemoryStore(index={"embed": letter_embed, "dims": 26})
print("index_config =", vector_store.index_config)

for text in ["今天成都天气晴", "User name is Bob", "User likes Python"]:
    vector_store.put(ns, str(uuid.uuid4()), {"value": text})

print("\n配了 index 后 query='Bob likes python'：")
for item in vector_store.search(ns, query="Bob likes python"):
    print(f"  score={item.score:.4f}  {item.value['value']}")

print("\n加 limit=1 才只拿最相关那条：")
for item in vector_store.search(ns, query="Bob likes python", limit=1):
    print(f"  score={item.score:.4f}  {item.value['value']}")


index_config = {'embed': <function letter_embed at 0x000002270E7D9EE0>, 'dims': 26, '__tokenized_fields': [('$', '$')]}

配了 index 后 query='Bob likes python'：
  score=0.6667  User likes Python
  score=0.5833  User name is Bob
  score=0.2108  今天成都天气晴

加 limit=1 才只拿最相关那条：
  score=0.6667  User likes Python


In [9]:
demo = InMemoryStore()
demo_ns = ("memories", "u_1")
demo.put(demo_ns, "profile", {"name": "Bob"})
demo.put(demo_ns, "pref:lang", {"value": "Python"})

print("A. 用同一个 key 再写一次（覆盖）：")
demo.put(demo_ns, "profile", {"name": "Bob", "city": "Chengdu"})
print("   get('profile').value =", demo.get(demo_ns, "profile").value)

print("\nB. 每次都用新 uuid（课堂 test3 的写法）：")
for _ in range(3):
    demo.put(demo_ns, str(uuid.uuid4()), {"value": "User likes Python"})
items = demo.search(demo_ns, limit=100)
print(f"   命名空间下一共 {len(items)} 条：")
for it in items:
    print(f"     key={it.key[:8]:8s} value={it.value}")
print("   同一个事实被存了 3 遍 → 检索结果重复，还会互相抢分数")


A. 用同一个 key 再写一次（覆盖）：
   get('profile').value = {'name': 'Bob', 'city': 'Chengdu'}

B. 每次都用新 uuid（课堂 test3 的写法）：
   命名空间下一共 5 条：
     key=profile  value={'name': 'Bob', 'city': 'Chengdu'}
     key=pref:lan value={'value': 'Python'}
     key=1db09268 value={'value': 'User likes Python'}
     key=fa811028 value={'value': 'User likes Python'}
     key=62d3d1e6 value={'value': 'User likes Python'}
   同一个事实被存了 3 遍 → 检索结果重复，还会互相抢分数


### 知识点解析 2：Store 的三个坑

| # | 坑 | 实测证据 | 正确做法 |
|---|---|---|---|
| 1 | 默认 `InMemoryStore()` **不做向量检索** | `index_config=None`、`embeddings=None`，检索结果 `score=None`，query 完全不起作用 | 要语义检索必须 `InMemoryStore(index={"embed": ..., "dims": n})` |
| 2 | `search` **不设相关度阈值**，只有 `limit`（默认 10） | 配好 index 后 3 条也全返回，只是排序变了 | 自己传 `limit=`；需要过滤分数就取出后按 `score` 筛 |
| 3 | `put` 的 key 自己给：同 key 覆盖、新 key 新增 | 上面第 B 段把同一事实写成 3 条 | 稳定的事实用**固定 key**（`profile`、`pref:lang`），多值事实才用 uuid |

补充两条：

- `InMemoryStore` 存在进程内，**进程一退就没了**；生产换 `PostgresStore` / `RedisStore`（接口一致）。
- `namespace` 是元组，惯例第一段写 `"memories"`（分区名），第二段写 `user_id`（隔离维度）。**别把 thread_id 放进去**——那样记忆又变成一次会话一份了。


## 3. 在节点里拿到 Store

节点想用 Store，靠**参数名**注入：签名里写 `store: BaseStore`，编译时 `compile(store=...)`。
同理还有 `config`、`runtime`——都是按名字识别的，变量名写错就注入不到（不会报错，只会是 `None`）。


In [10]:
from langgraph.store.base import BaseStore
from langgraph.checkpoint.memory import InMemorySaver

mem = InMemoryStore()
saver = InMemorySaver()


def node_with_store(state: MessagesState, store: BaseStore):
    namespace = ("memories", "user_1")
    store.put(namespace, "note:name", {"value": "User name is Bob"})     # 固定 key
    store.put(namespace, "note:lang", {"value": "User likes Python"})    # 固定 key
    hits = store.search(namespace, query="Bob likes python")
    return {"messages": [f"读到 {len(hits)} 条记忆：{[h.value['value'] for h in hits]}"]}


b = StateGraph(MessagesState)
b.add_node("recall", node_with_store)
b.add_edge(START, "recall")
b.add_edge("recall", END)
graph3 = b.compile(store=mem, checkpointer=saver)      # store 不配 checkpointer 也能用

for i in (1, 2):
    out = graph3.invoke({"messages": [("user", f"第{i}轮")]}, {"configurable": {"thread_id": "t_store"}})
    print(f"第{i}轮 → {out['messages'][-1].content}")

print("\n最终命名空间条目数 =", len(mem.search(("memories", "user_1"), limit=100)),
      "（固定 key，跑两次也不会膨胀）")


第1轮 → 读到 2 条记忆：['User name is Bob', 'User likes Python']
第2轮 → 读到 2 条记忆：['User name is Bob', 'User likes Python']

最终命名空间条目数 = 2 （固定 key，跑两次也不会膨胀）


In [11]:
b2 = StateGraph(MessagesState)
b2.add_node("recall", node_with_store)
b2.add_edge(START, "recall")
b2.add_edge("recall", END)
forgetful = b2.compile()                              # 忘了 store=

try:
    forgetful.invoke({"messages": []})
except AttributeError as e:
    print("编译阶段一声不吭，真用到才炸：")
    print(" ", type(e).__name__ + ":", e)


编译阶段一声不吭，真用到才炸：
  AttributeError: 'NoneType' object has no attribute 'put'


### 知识点解析 3

| 关注点 | 说明 |
|---|---|
| 注入方式 | 靠**参数名**：`store` / `config` / `runtime`；名字不匹配 → 参数是 `None` |
| 编译阶段校验 | **没有校验**：`compile()` 不传 `store` 也能成功，节点一用就 `AttributeError: 'NoneType' object has no attribute 'put'` |
| store 与 checkpointer | 相互独立：只要 store 可以不带 checkpointer；反之亦然 |
| 课堂 `test3.py` 的问题 | 节点里每次 `put(uuid4(), ...)` 两条 → 每 invoke 一次多两条，检索结果越来越脏 |

**推论**：`compile(store=...)` 这种「忘了也不报错」的设计，最好在项目里写一条烟雾测试——建图后立刻 invoke 一次，让 `None` 尽早暴露。


## 4. Runtime 与 Context：把「业务身份」传进去

`thread_id` 只回答「这是哪一段对话」，回答不了「这是谁」。后者用 **Context** 传：

| ID | 谁在用 | 是否落盘 | 典型用途 |
|---|---|---|---|
| `thread_id` | checkpointer | 是（会话快照） | 多轮对话、断点恢复、时间旅行 |
| `context.user_id` | 你的业务代码 / Store 命名空间 | 否（每次调用现传） | 用户隔离、权限、个性化 |

节点签名写 `runtime: Runtime[Context]`，就能拿到 `runtime.context`、`runtime.store`、`runtime.config`；也可以在节点里调 `get_config()` 直接取当前配置。


In [12]:
from langgraph.runtime import Runtime
from langgraph.config import get_config


@dataclass
class Context:
    user_id: str
    channel: str = "cli"


def who_am_i_node(state: MessagesState, runtime: Runtime[Context]):
    cfg = get_config()
    namespace = ("memories", runtime.context.user_id)          # 记忆按「人」隔离
    runtime.store.put(namespace, "last_seen", {"value": runtime.context.user_id})
    hit = runtime.store.get(namespace, "last_seen")
    return {"messages": [
        f"context.user_id={runtime.context.user_id!r} channel={runtime.context.channel!r}"
        f" | thread_id={cfg.get('configurable', {}).get('thread_id')!r}"
        f" | store 读回={hit.value if hit else None}"
    ]}


mem4 = InMemoryStore()
cfa = StateGraph(MessagesState)
cfa.add_node("who", who_am_i_node)
cfa.add_edge(START, "who")
cfa.add_edge("who", END)
graph4 = cfa.compile(store=mem4, checkpointer=InMemorySaver())

for tid, uid in [("chat-001", "user_1"), ("chat-002", "user_1"), ("chat-003", "user_2")]:
    out = graph4.invoke({"messages": [("user", "hi")]},
                        {"configurable": {"thread_id": tid}},
                        context=Context(user_id=uid))
    print(f"[thread={tid}] {out['messages'][-1].content}")

print("\n各用户命名空间："
      f"user_1 有 {len(mem4.search(('memories', 'user_1'), limit=100))} 条，"
      f"user_2 有 {len(mem4.search(('memories', 'user_2'), limit=100))} 条")


[thread=chat-001] context.user_id='user_1' channel='cli' | thread_id='chat-001' | store 读回={'value': 'user_1'}
[thread=chat-002] context.user_id='user_1' channel='cli' | thread_id='chat-002' | store 读回={'value': 'user_1'}
[thread=chat-003] context.user_id='user_2' channel='cli' | thread_id='chat-003' | store 读回={'value': 'user_2'}

各用户命名空间：user_1 有 1 条，user_2 有 1 条


In [13]:
try:
    graph4.invoke({"messages": [("user", "hi")]}, {"configurable": {"thread_id": "no-ctx"}})
except AttributeError as e:
    print("忘记传 context →", type(e).__name__ + ":", e)
    print("\n也就是说：Context 不会自动带默认值，runtime.context 直接是 None。")


忘记传 context → AttributeError: 'NoneType' object has no attribute 'user_id'

也就是说：Context 不会自动带默认值，runtime.context 直接是 None。


### 知识点解析 4

| 关注点 | 说明 |
|---|---|
| Context 写法 | `@dataclass` 最简（也可用 `TypedDict` / pydantic）；字段自己定，如 `user_id`、`channel`、`tenant` |
| 怎么传 | `graph.invoke(..., context=Context(user_id="user_1"))`；**每次调用都要传** |
| 落不落盘 | 不落盘。它属于「本次调用的输入」，不进 checkpoint |
| 不传会怎样 | `runtime.context` 是 `None`，一取字段就 `AttributeError: 'NoneType' object has no attribute 'user_id'` |
| 和 thread_id 的分工 | 上面输出里 `user_1` 跨了两个 thread 仍共享同一份记忆；`user_2` 一个 thread 就独立一份 |

> 记住这个组合：**Store 的 namespace 用 `context.user_id`，checkpointer 用 `thread_id`**。前者按人聚合，后者按会话隔离。


## 5. ToolRuntime：工具里也能拿到运行时，而且模型看不见

工具签名里写 `runtime: ToolRuntime[Context]`，工具内部就能读 `runtime.context` / `runtime.store` / `runtime.config` / `runtime.state` / `runtime.tool_call_id`。

**最关键的一点**：这个参数**不会出现在给模型的工具 schema 里**。模型看不到、也就伪造不了 `user_id`——所以「只能查自己的订单」这类校验必须写在工具内部（或用 `ToolRuntime` 取身份），而不是做成 `user_id` 参数让模型填。

```python
@tool
def my_orders(runtime: ToolRuntime[Context]) -> str:   # 模型只能看到 0 个参数
    uid = runtime.context.user_id                      # 身份来自运行时，模型给不了
```


In [14]:
from langchain.tools import ToolRuntime


@tool
def my_orders(runtime: ToolRuntime[Context]) -> str:
    """查询当前用户自己的订单列表"""
    uid = runtime.context.user_id
    return f"用户 {uid} 的订单：{ORDER_DB.get(uid, [])}"


@tool
def others_orders(target_user: str, runtime: ToolRuntime[Context]) -> str:
    """按用户 id 查询任意用户的订单（需要管理员权限）"""
    if runtime.context.user_id != "admin":
        return f"拒绝：{runtime.context.user_id} 不是管理员，无权查看 {target_user} 的订单"
    return f"{target_user} 的订单：{ORDER_DB.get(target_user, [])}"


ORDER_DB = {
    "user_1": [{"id": "A1001", "amount": 299}],
    "user_2": [{"id": "B2002", "amount": 88}],
}

bound = model.bind_tools([my_orders, others_orders])
print("模型实际看到的工具定义：")
for spec in bound.kwargs["tools"]:
    fn = spec["function"]
    params = list(fn["parameters"].get("properties", {}))
    print(f"  {fn['name']:14s} 参数={str(params):22s} required={fn['parameters'].get('required')}")


模型实际看到的工具定义：
  my_orders      参数=[]                     required=None
  others_orders  参数=['target_user']        required=['target_user']


In [15]:
def call_tool(single_tool, name: str, args: dict, user_id: str) -> str:
    """不经模型，直接用 ToolNode 触发一次工具调用，验证运行时注入。"""
    tb = StateGraph(MessagesState)
    tb.add_node("tool_node", ToolNode([single_tool]))
    tb.add_edge(START, "tool_node")
    tb.add_edge("tool_node", END)
    tg = tb.compile(checkpointer=InMemorySaver())
    out = tg.invoke(
        {"messages": [HumanMessage("x"),
                      AIMessage(content="", tool_calls=[{"name": name, "args": args, "id": "c1"}])]},
        {"configurable": {"thread_id": f"t-{user_id}-{name}"}},
        context=Context(user_id=user_id),
    )
    return out["messages"][-1].content


print("user_1 查自己的 :", call_tool(my_orders, "my_orders", {}, "user_1"))
print("user_1 查别人的 :", call_tool(others_orders, "others_orders", {"target_user": "user_2"}, "user_1"))
print("admin  查别人的 :", call_tool(others_orders, "others_orders", {"target_user": "user_2"}, "admin"))

print("\n注意：同一个 tool_call_id 里，模型的 args 只有 target_user，user_id 完全由运行时决定。")


user_1 查自己的 : 用户 user_1 的订单：[{'id': 'A1001', 'amount': 299}]
user_1 查别人的 : 拒绝：user_1 不是管理员，无权查看 user_2 的订单
admin  查别人的 : user_2 的订单：[{'id': 'B2002', 'amount': 88}]

注意：同一个 tool_call_id 里，模型的 args 只有 target_user，user_id 完全由运行时决定。


### 知识点解析 5

| 关注点 | 说明 |
|---|---|
| 导入路径 | 真实定义在 `langgraph.prebuilt.tool_node.ToolRuntime`，`langchain.tools` 只是 re-export |
| 泛型参数 | 其实是**两个**：`ToolRuntime[Context, State]`；只写一个时状态类型默认 `dict` |
| 可用字段 | `context` / `store` / `config` / `state` / `tool_call_id` / `stream_writer` |
| 安全意义 | 参数不进 schema（上面输出里 `my_orders` 的 `参数=[]`），模型无法伪造身份 |
| 和节点版区别 | 节点用 `Runtime[Context]`，工具用 `ToolRuntime[Context]`；后者多了 `state`、`tool_call_id` |

课堂 `test5.py` 里的两个小毛病：

| 写法 | 问题 | 建议 |
|---|---|---|
| `def get_weather(...) -> WeatherResponse:` 但 `return "The weather in ..."` | 返回类型注解与实际不符，pydantic 类被当成「结构化返回值」承诺给框架 | 返回 `-> str`，pydantic 只用来做 `args_schema` |
| `class WeatherResponse(BaseModel)` 既当参数 schema 又当返回类型 | 名字和用途对不上，读代码的人会误解 | 参数 schema 命名成 `WeatherArgs` 之类 |


## 6. 流式输出：`stream_mode` 的三种形状

同样一个 `graph.stream(...)`，`stream_mode` 决定你拿到的是什么：

| 模式 | 每次吐出 | 元素形状 | 典型用途 |
|---|---|---|---|
| `messages` | 模型产出的 **token 级 chunk**（含 `ToolMessage` 整条） | `(message_chunk, metadata)` | 打字机效果、前端逐字渲染 |
| `values` | 每个 superstep 结束的**完整 state** | `{"messages": [...]}` | 看状态一步步长出来 |
| `updates` | 该节点自己的**返回值** | `{"节点名": {...}}` | 调试路由、看清谁干了什么 |

下面三格分别跑同一个问题（真模型），第四格演示多模式复用。


In [16]:
print("=== stream_mode='messages'（真模型逐 token 吐）=== ")
empty_frames = 0
for chunk, meta in graph1.stream({"messages": [("user", QUESTION)]}, stream_mode="messages"):
    if isinstance(chunk, ToolMessage):                      # 工具结果是整条，不逐字
        print(f"\n[工具结果整条 @ {meta['langgraph_node']}] {chunk.content}")
    elif chunk.content:
        print(chunk.content, end="", flush=True)
    else:
        empty_frames += 1                                   # 工具调用帧 content 为空
print(f"\n\n（另有 {empty_frames} 个空 content 帧：工具调用指令就藏在这些帧里，前端必须过滤）")


=== stream_mode='messages'（真模型逐 token 吐）=== 

[工具结果整条 @ tool_node] 东京 今天晴，25℃
东京今天天气晴朗，气温为25℃。

（另有 13 个空 content 帧：工具调用指令就藏在这些帧里，前端必须过滤）


In [17]:
print("=== stream_mode='values'（每步一份完整 state）===")
for i, state in enumerate(graph1.stream({"messages": [("user", QUESTION)]}, stream_mode="values"), 1):
    last = state["messages"][-1]
    print(f"第{i}步 | 累计 {len(state['messages'])} 条 | 末条 {type(last).__name__:11s} "
          f"| {str(last.content)[:38]!r}")


=== stream_mode='values'（每步一份完整 state）===
第1步 | 累计 1 条 | 末条 HumanMessage | '请调用工具查一下东京今天的天气，再用一句话总结'
第2步 | 累计 2 条 | 末条 AIMessage   | ''
第3步 | 累计 3 条 | 末条 ToolMessage | '东京 今天晴，25℃'
第4步 | 累计 4 条 | 末条 AIMessage   | '东京今天天气晴朗，气温为25℃。'


In [18]:
print("=== stream_mode='updates'（每个节点自己的返回值）===")
for step in graph1.stream({"messages": [("user", QUESTION)]}, stream_mode="updates"):
    for node, update in step.items():
        kinds = [type(m).__name__ for m in update.get("messages", [])]
        print(f"  {node:10s} → 返回键 {list(update)} | 消息类型 {kinds}")


=== stream_mode='updates'（每个节点自己的返回值）===
  chat       → 返回键 ['messages'] | 消息类型 ['AIMessage']
  tool_node  → 返回键 ['messages'] | 消息类型 ['ToolMessage']
  chat       → 返回键 ['messages'] | 消息类型 ['AIMessage']


In [19]:
print("=== stream_mode=['updates','messages'] （多模式复用）===")
# 本格换成剧本模型，保证一定能走完「chat → tool_node → chat」，输出稳定可对比
mb = StateGraph(MessagesState)
mb.add_node("chat", make_scripted_chat("get_weather", {"city": "Tokyo"}))
mb.add_node("tool_node", ToolNode(tools))
mb.add_edge(START, "chat")
mb.add_conditional_edges(
    "chat", lambda s: "tool_node" if s["messages"][-1].tool_calls else END, ["tool_node", END])
mb.add_edge("tool_node", "chat")
graph_multi = mb.compile()

for mode, payload in graph_multi.stream({"messages": [("user", "东京天气")]},
                                        stream_mode=["updates", "messages"]):
    if mode == "updates":
        node_name = list(payload)[0]
        kinds = [type(m).__name__ for m in payload[node_name]["messages"]]
        print(f"[updates ] 节点 {node_name:10s} 返回消息类型 {kinds}")
    else:
        chunk, meta = payload
        label = "工具结果" if isinstance(chunk, ToolMessage) else "模型输出"
        print(f"[messages] {meta['langgraph_node']:10s} {label} {chunk.content!r}")


=== stream_mode=['updates','messages'] （多模式复用）===
[messages] chat       模型输出 ''
[updates ] 节点 chat       返回消息类型 ['AIMessage']
[messages] tool_node  工具结果 'Tokyo 今天晴，25℃'
[updates ] 节点 tool_node  返回消息类型 ['ToolMessage']
[messages] chat       模型输出 '（剧本模型）已按工具结果作答。工具返回：Tokyo 今天晴，25℃'
[updates ] 节点 chat       返回消息类型 ['AIMessage']


### 知识点解析 6

| # | 坑 | 说明 / 正确做法 |
|---|---|---|
| 1 | `messages` 模式的 chunk 是 **`AIMessageChunk`**，`.content` 是增量片段 | 要完整文本得自己 `full += chunk`；不能把每个 chunk 当完整句子 |
| 2 | 工具调用帧的 `content` 是**空字符串** | 前端不判空会刷出一堆空行；按 `if chunk.content:` 过滤 |
| 3 | 判断「是哪个节点吐的」 | 看 `metadata["langgraph_node"]`，不要靠猜 |
| 4 | `values` 每步都给**完整历史** | 长对话时输出量平方级增长，只适合调试，不适合给前端 |
| 5 | 多模式复用返回 **`(mode, payload)` 元组** | 顺序不保证，必须按 `mode` 分支处理（上面输出里 messages 先到） |
| 6 | `ToolMessage` 在 `messages` 模式下是**整条**出现 | 它不逐字生成，所以别指望工具结果有打字机效果（第 16 格用 `isinstance` 单独标了出来） |
| 7 | 真模型**可能不调工具** | 第 19 格换成剧本模型，保证「chat → tool_node → chat」三步都在，输出可对比 |

补充：除了这三种，还有 `stream_mode="custom"`（节点里用 `get_stream_writer()` 自己写事件）和 `"debug"`。前端对接优先 `messages`，调试优先 `updates`。


## 7. 静态断点：`interrupt_before`

编译时声明 `interrupt_before=["tool_node"]`，图跑到这一步就停住。停了以后有两条路：

1. **放行**：`graph.stream(None, config)` —— 输入 `None` 就是「继续」。
2. **否决**：`graph.update_state(config, {"messages": [ToolMessage(...)]}, as_node="tool_node")` —— 伪造一条工具结果，假装 `tool_node` 已经跑完。

为什么否决时**必须**补 `ToolMessage`？因为 state 末尾那条 `AIMessage` 带着 `tool_calls`，下一次调模型时接口要求每个 `tool_call_id` 都有对应结果，缺了直接 400。

下面两格分别走这两条路（用剧本模型保证确定性）。


In [20]:
def build_paused_graph():
    b = StateGraph(MessagesState)
    b.add_node("chat", make_scripted_chat())        # 剧本模型：稳定产出一次 tool_call
    b.add_node("tool_node", ToolNode(tools))
    b.add_edge(START, "chat")
    b.add_conditional_edges(
        "chat", lambda s: "tool_node" if s["messages"][-1].tool_calls else END, ["tool_node", END])
    b.add_edge("tool_node", "chat")
    return b.compile(checkpointer=InMemorySaver(), interrupt_before=["tool_node"])


PAYLOAD = {"messages": [("user", "东京今天天气怎么样？")]}

g7 = build_paused_graph()
cfg7 = {"configurable": {"thread_id": "pause-approve"}}
for _ in g7.stream(PAYLOAD, cfg7, stream_mode="values"):
    pass

snap = g7.get_state(cfg7)
print("停在 next =", snap.next)
print("待执行的 tool_calls =", snap.values["messages"][-1].tool_calls)

print("\n== 选择放行：输入 None ==")
for _ in g7.stream(None, cfg7, stream_mode="values"):
    pass
print("最终回答：", g7.get_state(cfg7).values["messages"][-1].content)


停在 next = ('tool_node',)
待执行的 tool_calls = [{'name': 'get_weather', 'args': {'city': 'Tokyo'}, 'id': 'call_1', 'type': 'tool_call'}]

== 选择放行：输入 None ==
最终回答： （剧本模型）已按工具结果作答。工具返回：Tokyo 今天晴，25℃


In [21]:
g7b = build_paused_graph()
cfg7b = {"configurable": {"thread_id": "pause-reject"}}
for _ in g7b.stream(PAYLOAD, cfg7b, stream_mode="values"):
    pass

snap_b = g7b.get_state(cfg7b)
tool_call_id = snap_b.values["messages"][-1].tool_calls[0]["id"]

reject_cfg = g7b.update_state(
    cfg7b,
    {"messages": [ToolMessage(content="用户拒绝执行该工具，请直接回答", tool_call_id=tool_call_id)]},
    as_node="tool_node",          # 假装这条更新是 tool_node 跑完产生的
)
print("update_state 返回的是新的 config：", reject_cfg)

for _ in g7b.stream(None, cfg7b, stream_mode="values"):
    pass

final = g7b.get_state(cfg7b).values["messages"]
print("\n最终回答：", final[-1].content)
print("消息序列：", [type(m).__name__ for m in final])


update_state 返回的是新的 config： {'configurable': {'thread_id': 'pause-reject', 'checkpoint_ns': '', 'checkpoint_id': '1f1db2e4-9105-682a-8002-5fcb0cd636f2'}}

最终回答： （剧本模型）已按工具结果作答。工具返回：用户拒绝执行该工具，请直接回答
消息序列： ['HumanMessage', 'AIMessage', 'ToolMessage', 'AIMessage']


### 知识点解析 7

| 关注点 | 说明 |
|---|---|
| 停顿点 | `snapshot.next == ("tool_node",)`，表示「下一步该执行 tool_node，但还没执行」 |
| 放行的写法 | `graph.stream(None, config)`；`invoke(None, config)` 同理 |
| 否决的写法 | `update_state(config, {"messages": [ToolMessage(...)]}, as_node="tool_node")` |
| `as_node` 的作用 | 告诉框架「这条更新是哪一步产生的」，决定后续从哪个节点的出边走 |
| 为什么必须补 ToolMessage | `AIMessage.tool_calls` 与 `ToolMessage.tool_call_id` 必须成对，否则下次调模型报 400 |
| 必须配 checkpointer | 不然没有 `snapshot`、`next`，也没法 resume |
| 与下一节的区别 | 这里是**节点执行前**停，payload 只能靠 state；`interrupt()` 能停在一行中间，还能带问题问人 |

课堂 `test7.py` 用 `input()` 做交互——在 notebook / Web 服务里会直接卡死。正确姿势是把「批不批」变成外部事件（前端按钮、消息队列），不能阻塞在图里。


## 8. 动态中断：工具内 `interrupt()`

`interrupt_before` 只能「进节点前停」，也没法带问题去问人。要停在**中间某一行**，用 `interrupt()`：

```python
decision = interrupt({"question": "是否批准这笔退款？", "amount": 199})
```

| | `interrupt_before=[...]` | 工具内 `interrupt(payload)` |
|---|---|---|
| 声明时机 | 编译期，静态 | 运行期，走到那一行才触发 |
| 能否带信息问人 | 不能 | 能，payload 会出现在 `snapshot.interrupts` |
| 恢复方式 | `stream(None, cfg)` | `stream(Command(resume=值), cfg)`，`interrupt()` 返回该值 |
| 恢复时是否重跑 | 不重跑（节点本来没执行过） | **会重跑一次**（下一格实测） |


In [22]:
from langgraph.types import interrupt, Command

enter_count = {"n": 0}


@tool
def refund(amount: int) -> str:
    """发起退款，属于高风险操作，必须人工确认"""
    enter_count["n"] += 1
    print(f"    [refund 第 {enter_count['n']} 次进入函数]")
    decision = interrupt({"question": "是否批准这笔退款？", "amount": amount})
    return f"人工结论={decision}"


def chat_refund(state: MessagesState):
    if any(isinstance(m, ToolMessage) for m in state["messages"]):
        return {"messages": [AIMessage(content="已按人工结论处理完毕。")]}
    return {"messages": [AIMessage(content="", tool_calls=[
        {"name": "refund", "args": {"amount": 199}, "id": "refund-1"}])]}


b8 = StateGraph(MessagesState)
b8.add_node("chat", chat_refund)
b8.add_node("tool_node", ToolNode([refund]))
b8.add_edge(START, "chat")
b8.add_conditional_edges(
    "chat", lambda s: "tool_node" if s["messages"][-1].tool_calls else END, ["tool_node", END])
b8.add_edge("tool_node", "chat")
g8 = b8.compile(checkpointer=InMemorySaver())
cfg8 = {"configurable": {"thread_id": "interrupt-1"}}

g8.invoke({"messages": [HumanMessage("我要退款 199 元")]}, cfg8)
snap8 = g8.get_state(cfg8)
print("挂起后 next =", snap8.next)
print("interrupts  =", snap8.interrupts)

out8 = g8.invoke(Command(resume="批准"), cfg8)
print("\n恢复后消息序列：")
for m in out8["messages"]:
    print(f"  {type(m).__name__:11s} {str(m.content)[:60]!r}")


    [refund 第 1 次进入函数]
挂起后 next = ('tool_node',)
interrupts  = (Interrupt(value={'question': '是否批准这笔退款？', 'amount': 199}, id='c54981276499c88107a0a10ec488e3fe', response_schema=None),)
    [refund 第 2 次进入函数]

恢复后消息序列：
  HumanMessage '我要退款 199 元'
  AIMessage   ''
  ToolMessage '人工结论=批准'
  AIMessage   '已按人工结论处理完毕。'


In [23]:
print("refund 工具总共被进入", enter_count["n"], "次")
print(f"分解：挂起前 1 次 + 恢复时重放 {enter_count['n'] - 1} 次")
print("结论：interrupt() 之前的代码会被重复执行，副作用必须幂等或挪到 interrupt 之后。")


refund 工具总共被进入 2 次
分解：挂起前 1 次 + 恢复时重放 1 次
结论：interrupt() 之前的代码会被重复执行，副作用必须幂等或挪到 interrupt 之后。


### 知识点解析 8

| # | 坑 | 实测 / 说明 |
|---|---|---|
| 1 | **恢复会重跑节点** | 实测：`ToolNode` 里的工具函数共进入 **2 次**（挂起 1 次 + 恢复重放 1 次）；普通节点同样进入 2 次 |
| 2 | 副作用位置 | 写库、扣费、发消息这类必须放在 `interrupt()` **之后**，或做成幂等（带业务唯一键） |
| 3 | 必须配 checkpointer | 没有检查点就无处记录「停在哪一行」，`interrupt()` 无法工作 |
| 4 | 非确定性代码 | `time.time()`、`uuid4()`、随机数放在 interrupt 前，重跑会得到不同值，进而让状态对不上 |
| 5 | 恢复才会往前走 | 挂起后必须用 `Command(resume=...)`；直接 `invoke(None, cfg)` 拿不到值（`interrupt()` 返回值要由 resume 提供） |
| 6 | `interrupts` 里是什么 | `Interrupt(value=payload, id=...)`，`id` 与调用点的位置绑定，同一函数里多个 `interrupt()` 靠它区分 |

**适用场景**：高风险工具（退款、转账、删数据）人工确认；信息不足时向用户追问；长流程中间做策略审批。

> 设计口诀：**能进 checkpoint 的才放 state，需要人给答案的走 resume 值；两者都不能靠全局变量传递。**


## 9. 时间旅行：重放与分叉

有了检查点，历史就不是流水账，而是**可以跳回去的坐标**：

| 操作 | API | 结果 |
|---|---|---|
| 看历史 | `graph.get_state_history(config)` | 快照列表，**最新在前** |
| 重放 | `graph.invoke(None, snapshot.config)` | 从那个点原样再跑一遍，历史继续追加 |
| 分叉 | `fork = graph.update_state(snapshot.config, 新写入, as_node=...)` → `invoke(None, fork)` | 走出一条新分支，旧分支保留 |


In [24]:
def build_full_graph():
    b = StateGraph(MessagesState)
    b.add_node("chat", make_scripted_chat())
    b.add_node("tool_node", ToolNode(tools))
    b.add_edge(START, "chat")
    b.add_conditional_edges(
        "chat", lambda s: "tool_node" if s["messages"][-1].tool_calls else END, ["tool_node", END])
    b.add_edge("tool_node", "chat")
    return b.compile(checkpointer=InMemorySaver())


g9 = build_full_graph()
cfg9 = {"configurable": {"thread_id": "tt-1"}}
g9.invoke({"messages": [HumanMessage("东京今天天气怎么样？")]}, cfg9)

history = list(g9.get_state_history(cfg9))
print(f"一次完整运行产生 {len(history)} 个快照（倒序，最新在前）：")
for i, s in enumerate(history):
    ckpt = s.config["configurable"]["checkpoint_id"][-8:]
    print(f"  #{i} next={str(s.next):15s} 消息数={len(s.values['messages'])} checkpoint=...{ckpt}")


一次完整运行产生 5 个快照（倒序，最新在前）：
  #0 next=()              消息数=4 checkpoint=...80190fbf
  #1 next=('chat',)       消息数=3 checkpoint=...e1d5c20c
  #2 next=('tool_node',)  消息数=2 checkpoint=...9e8565ea
  #3 next=('chat',)       消息数=1 checkpoint=...d35e79ce
  #4 next=('__start__',)  消息数=0 checkpoint=...f89e6668


In [25]:
target = next(s for s in history if s.next == ("tool_node",))
print("选中回放点：next =", target.next, "| 消息数 =", len(target.values["messages"]))
print("该点的末条消息：", type(target.values["messages"][-1]).__name__,
      target.values["messages"][-1].tool_calls)

before = len(list(g9.get_state_history(cfg9)))
g9.invoke(None, target.config)                       # 从这里继续，等于原路重放
after = len(list(g9.get_state_history(cfg9)))

print(f"\n重放后快照数 {before} → {after}")
print("重放后的最终回答：", g9.get_state(cfg9).values["messages"][-1].content)


选中回放点：next = ('tool_node',) | 消息数 = 2
该点的末条消息： AIMessage [{'name': 'get_weather', 'args': {'city': 'Tokyo'}, 'id': 'call_1', 'type': 'tool_call'}]

重放后快照数 5 → 8
重放后的最终回答： （剧本模型）已按工具结果作答。工具返回：Tokyo 今天晴，25℃


In [26]:
history2 = list(g9.get_state_history(cfg9))
target2 = next(s for s in history2 if s.next == ("tool_node",))
tool_call_id2 = target2.values["messages"][-1].tool_calls[0]["id"]

before2 = len(history2)
fork_cfg = g9.update_state(
    target2.config,
    {"messages": [ToolMessage(content="【人工改判】工具结果不可信，请回答不知道",
                              tool_call_id=tool_call_id2)]},
    as_node="tool_node",
)
print("分叉点 checkpoint = ...", fork_cfg["configurable"]["checkpoint_id"][-8:], sep="")

g9.invoke(None, fork_cfg)
print("分叉后的最终回答：", g9.get_state(cfg9).values["messages"][-1].content)
print(f"分叉后快照数 {before2} → {len(list(g9.get_state_history(cfg9)))}（旧分支没有消失）")


分叉点 checkpoint = ...49ce23ab
分叉后的最终回答： （剧本模型）已按工具结果作答。工具返回：【人工改判】工具结果不可信，请回答不知道
分叉后快照数 8 → 10（旧分支没有消失）


### 知识点解析 9

| # | 坑 | 实测 / 说明 |
|---|---|---|
| 1 | 历史是**倒序** | `history[0]` 是最新快照；别拿 `[-1]` 当「上一次」 |
| 2 | `next` 的含义 | `()` = 已跑完；`('tool_node',)` = 下一步该执行它；`('__start__',)` = 刚提交输入 |
| 3 | 同名节点会出现多次 | 循环图里 `('chat',)` 不止一个快照，要靠消息数 / checkpoint 区分 |
| 4 | `update_state` 返回的是 **config** | 带新的 `checkpoint_id`，必须拿它去 `invoke(None, fork_cfg)` 才能走到分叉上 |
| 5 | 旧分支不会被删 | 快照只增不减，本机实测 5 → 8 → 10 |
| 6 | 重放要传 `None` | 传新输入等于开新一轮，不是从历史点继续 |

**能拿它干什么**：调试时把「模型答错的那一步」跳回去改工具结果再跑；A/B 两种工具实现用同一段历史对比；线上事故复盘时还原每一步的快照。

> 注意成本：`InMemorySaver` 全量留在内存，长会话会越来越大。生产用 Redis/Postgres 检查点并设保留策略。


## 10. 收口：九个脚本 → 一张能力地图

| 脚本 | 讲的东西 | 一句话结论 | 本笔记对应 |
|---|---|---|---|
| test1 | `ToolNode` | 「申请调用」与「执行工具」拆两个节点，条件边闭合循环 | 第 1 节 |
| test2 | `InMemoryStore` | 默认不配 `index` 就**不是**语义检索 | 第 2 节 |
| test3 | 节点注入 store | 靠参数名注入，忘了 `compile(store=)` 不报错 | 第 3 节 |
| test4 | `Runtime[Context]` | `thread_id` 管会话，`context.user_id` 管身份 | 第 4 节 |
| test5 | `ToolRuntime` | 工具内取运行时，身份不进 schema，天然防伪造 | 第 5 节 |
| test6 | `stream_mode` | messages / values / updates 三种形状，用途完全不同 | 第 6 节 |
| test7 | `interrupt_before` | 节点前停顿；否决要伪造 `ToolMessage` 补位 | 第 7 节 |
| test8 | `interrupt()` | 停在中间一行；恢复会重跑，副作用要幂等 | 第 8 节 |
| test9 | `get_state_history` | 历史可重放可分叉，旧分支不删 | 第 9 节 |

### 全篇坑清单（按危险程度排序）

| 危险度 | 坑 | 后果 |
|---|---|---|
| 高 | `interrupt()` 之前写副作用 | 恢复时重复执行，重复退款/重复发消息 |
| 高 | `ToolRuntime` 的身份做成普通参数让模型填 | 模型可伪造 `user_id`，越权查数据 |
| 高 | 以为 `InMemoryStore()` 默认做语义检索 | 检索退化成「全量返回」，线上召回看着有结果其实全是噪声 |
| 中 | 节点里 `put(uuid4(), ...)` | 记忆无限膨胀、结果重复 |
| 中 | 同一节点同时挂静态边和条件边 | 路由语义含混，容易提前 END |
| 中 | `interrupt_before` + `input()` | 阻塞整个线程/服务 |
| 中 | `stream_mode="values"` 给前端 | 输出量平方级增长 |
| 低 | 返回类型注解写成 pydantic 类但实际返回 str | 误导调用方与框架 |
| 低 | API Key 复制到 9 个文件 | 泄漏面乘 9 |

### 待办（给下一轮学习）

1. 把这套「Store + Context + 权限校验」抽成一个可复用的 `build_agent()` 工厂，替换 9 个脚本里的重复样板。
2. 用 `PostgresStore` + `PostgresSaver` 重做第 2、9 节，验证跨进程持久化。
3. 给 `interrupt()` 场景加一个幂等键（业务单号），在恢复重跑时验证不会重复扣款。
4. 用 `stream_mode="custom"` + `get_stream_writer()` 做一套前端事件协议（进度、思考、工具调用、结果）。


## 自测题（答案在第 9、10 节的实测输出里）

1. `ToolNode` 执行工具后写回的 `ToolMessage`，靠什么字段和那条 `AIMessage` 对上？
2. 不配 `index` 的 `InMemoryStore`，`search(query=...)` 的返回值有什么特征？为什么危险？
3. 节点签名写了 `store: BaseStore` 却忘记 `compile(store=...)`，会在什么时候报错？
4. `thread_id` 和 `context.user_id` 各自决定什么？哪个会落盘？
5. 为什么「只能查自己订单」必须用 `ToolRuntime`，而不能给工具加一个 `user_id` 参数？
6. `stream_mode="messages"` 里工具调用帧的 `content` 是什么？前端怎么过滤？
7. `interrupt_before` 停下后，想直接否决，为什么必须补一条 `ToolMessage`？
8. `interrupt()` 恢复时为什么会「重跑」？哪些代码必须挪到它之后？
9. `get_state_history` 返回的列表里，`history[0]` 是最新还是最旧？`next=()` 表示什么？
10. `update_state` 的返回值是 state 还是 config？拿它做什么？

## 运行说明

| 项 | 说明 |
|---|---|
| 内核 | `langchain-learning (3.11.9)`（`week23/.venv`） |
| 启动目录 | `week24`（`find_lab_dir()` 需要向上找到 `week23/20261027`） |
| 网络 | 自检、第 1、5、6 节会调千问；其余各节离线可跑 |
| 想纯离线 | 把自检里的 `model` 换成 `ScriptedModel()` 即可，但第 6 节的 token 流会退化成「一条完整消息」 |
| 复现性 | 第 1、7、8、9 节用剧本模型，输出与本文件保存的一致 |
